In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone

import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:

utc_now_iso = datetime.now(timezone.utc).isoformat()
print(utc_now_iso)

2025-07-31T13:27:52.923761+00:00


In [3]:
base_url = "http://personal-laptop.local:8080/"


In [4]:
def create_url(base_url,first_part_endpoint,second_part_endpoint):
    return base_url + first_part_endpoint + second_part_endpoint
def getData(url):
    response = requests.get(url)

    # Check response status
    if response.status_code == 200:
        
        data = response.json()
        print(f"Received {len(data)} records.")
        return data
    else:
        print(f"Error: {response.status_code}")


def saveDataIntoDataFolder(data,data_file_name):
    script_dir = Path().resolve().parent
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    with open(file_path, 'w') as file:
        if isinstance(data, pd.DataFrame):
            print("It's a DataFrame")
            if  data.empty():
                print("No data to save.")
            else:
                data.to_json(file_path, orient='records', lines=False)             
                print(f"Data saved to {file_path}")

        else:  
            print("It's NOT a DataFrame.")    
            if not data:
                print("No data to save.")
            else:    
                json.dump(data, file)
                print(f"Data saved to {file_path}")

        
def getDataFromServerThenSaveThemIntoDataFolder(url,data_file_name):
    data = getData(url)
    saveDataIntoDataFolder(data,data_file_name)
    

def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None

def countPositionOfPollutantSource(df_userInputData_smallScale):
    df_unique_position_count = df_userInputData_smallScale.groupby(['front-wall', 'side-right-wall','back-wall','side-left-wall'], dropna=False).size().reset_index(name='count')
    return df_unique_position_count

    
        

In [5]:
def getTimeSeries(df_userInputData_smallScale):
    first_row= df_userInputData_smallScale.iloc[0]
    last_row = df_userInputData_smallScale.iloc[-1]
    start_time = first_row["epoch_ms"]
    end_time   = last_row["epoch_ms"] 
    first_part_endpoint = "timeSeriesEndpoints/"
    second_part_endpoint = "getTimeSeriesData?start=" + str(start_time) + '&end=' +str(end_time)
    file_name="TimeSeries"
    url = create_url(base_url,first_part_endpoint,second_part_endpoint)
    getDataFromServerThenSaveThemIntoDataFolder(url,file_name)
    df_timeSeries = loadDataFromFile(file_name)
    return df_timeSeries
def tranformTimeSeries(df_timeSeries):
    # 1. Select only the needed column
    df_timeSeriesExperiments = df_timeSeries[['timestamp', 'Id', 'BME680:breathVocEquivalent']].copy()
    # 2. Convert `timestamp` (UTC) to Europe/Athens tz-aware datetime
    df_timeSeriesExperiments['timestamp'] = (
        pd.to_datetime(df_timeSeriesExperiments['timestamp'], utc=True)
          .dt.tz_convert('Europe/Athens')
    )
    # 3. Create the two “Id=…:” columns
    
    df_timeSeriesExperiments['Id=1:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 1, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    df_timeSeriesExperiments['Id=2:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 2, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    
    # 4. Set timestamp as the row index
    #df_timeSeriesExperiments.set_index('timestamp', inplace=True)
    
    # 5. Drop the old Id+raw measurement columns
    df_timeSeriesExperiments = df_timeSeriesExperiments.drop(
        columns=['Id', 'BME680:breathVocEquivalent']
    )
    return df_timeSeriesExperiments

In [6]:
def sortUserIputDataToListBasedOfPosition(df_userInputData_smallScale):
    
    list_of_userInputData_smallScale_based_on_position = []
    
    for idx, row in df_unique_position_count.iterrows():
        mask = (
            (df_userInputData_smallScale['experimentState'] == "InsertingSourcePollutant") &
            (
                (df_userInputData_smallScale['front-wall'] == row['front-wall']) |
                (np.isnan(df_userInputData_smallScale['front-wall']) & np.isnan(row['front-wall']))
            ) &
            (
                (df_userInputData_smallScale['side-left-wall'] == row['side-left-wall']) |
                (np.isnan(df_userInputData_smallScale['side-left-wall']) & np.isnan(row['side-left-wall']))
            ) &
            (
                (df_userInputData_smallScale['back-wall'] == row['back-wall']) |
                (np.isnan(df_userInputData_smallScale['back-wall']) & np.isnan(row['back-wall']))
            ) &
            (
                (df_userInputData_smallScale['side-right-wall'] == row['side-right-wall']) |
                (np.isnan(df_userInputData_smallScale['side-right-wall']) & np.isnan(row['side-right-wall']))
            )
        )

        df_userInputData_individual_position = df_userInputData_smallScale[mask]
        list_of_userInputData_smallScale_based_on_position.append(df_userInputData_individual_position)

    return list_of_userInputData_smallScale_based_on_position


In [7]:
def print_time_series_based_of_position(index,df_userInputData_smallScale,df_timeSeriesExperiments,
                                        list_of_userInputData_smallScale_based_of_position,oldData = False):
    df_userInputData_smallScale_based_of_position = list_of_userInputData_smallScale_based_of_position[index]
    num_experiments = df_userInputData_smallScale_based_of_position.shape[0]
    fig, axes = plt.subplots(num_experiments, figsize=(14, 3 * num_experiments), sharex=False)
    if num_experiments == 1:
        axes = [axes]  # Wrap single Axes into a list
    title = f"BME680:breathVocEquivalent at the pollution source position to be "
    if (pd.isna(df_userInputData_smallScale_based_of_position.iloc[0]['front-wall'])==False):
       title =title + f"{df_userInputData_smallScale_based_of_position.iloc[0]['front-wall']} meters from the front wall, "
    if (pd.isna(df_userInputData_smallScale_based_of_position.iloc[0]['back-wall'])==False):
       title =title +f"{df_userInputData_smallScale_based_of_position.iloc[0]['back-wall']} meters from the back wall, "
    if (pd.isna(df_userInputData_smallScale_based_of_position.iloc[0]['side-right-wall'])==False):
       title =title+ f"{df_userInputData_smallScale_based_of_position.iloc[0]['side-right-wall']} meters from the side right wall, "
    if (pd.isna(df_userInputData_smallScale_based_of_position.iloc[0]['side-left-wall'])==False):
       title =title+ f"{df_userInputData_smallScale_based_of_position.iloc[0]['side-left-wall']} meters from the side left wall, "

    title =title +"at the room:"+df_userInputData_smallScale_based_of_position.iloc[0]["room"]   
    fig.suptitle(title)

    for i, (idx, row) in enumerate(df_userInputData_smallScale_based_of_position.iterrows()):
        # Filter the time series between start and end 
        if 'timestamp:Starting Experiment"' in df.columns :

        if 'column_name' in df.columns :

            
        if (oldData == False):
            starting_experiment_timestamp =  df_userInputData_smallScale.iloc[idx - 1]["timestamp_local"]         
        #create a pseudo start experiment 3 minutes before the source insertion    
        elif  (oldData == True): 
            
            starting_experiment_timestamp = df_userInputData_smallScale.iloc[idx]["timestamp_local"] - datetime.timedelta(minutes=3)
        removing_source_timestamp = df_userInputData_smallScale.iloc[idx + 1]["timestamp_local"]   
  
        inserting_source_timestamp_condition  = (df_timeSeriesExperiments["timestamp"] >= starting_experiment_timestamp )    
        removing_source_timestamp_condition =   (df_timeSeriesExperiments["timestamp"] <= removing_source_timestamp  )

        data = df_timeSeriesExperiments[removing_source_timestamp_condition & inserting_source_timestamp_condition]         
        # Plot VOC data
        sns.lineplot(
            data=data, 
            x="timestamp",
            y='Id=1:BME680:breathVocEquivalent',
            label="Id=1",
            marker='o',
            ax=axes[i]
        )
        sns.lineplot(
            data=data,
            x="timestamp",
            y='Id=2:BME680:breathVocEquivalent',
            label="Id=2",
            marker='s',
            ax=axes[i]
        )
        print()
        subtitle="Στοιχεία για το μενωμένο πείραμα:"
        subtitle= subtitle+"Αντικείμενο που χρησιμοποιείται¨"+row["item-used"]+", "
        if (pd.isna(row["are-doors-opened"]) == False):
            subtitle = subtitle + "Οι πόρτες είναι ανοιχτές, "
        if (pd.isna(row["are-fans-on"]) == False):
            subtitle = subtitle + "Οι ανεμιστήρες είναι ενεργοποιημένοι, "
        if (pd.isna(row["are-people-inside"]) == False):
            subtitle = subtitle + "Βρίσκονται άνθρωποι μέσα, "
        if (pd.isna(row["are-windows-opened"]) == False):
            subtitle = subtitle + "Τα παράθυρα είναι ανοιχτά, "
        if (pd.isna(row["notes"])== False):
            subtitle= subtitle + "Σημειώσεις:"+ row["notes"]
        axes[i].set_title(subtitle);

        # Draw vertical line at inserting source timestamp
        inserting_source = df_userInputData_smallScale.iloc[idx]["timestamp_local"]
        if inserting_source:
            axes[i].axvline(
                inserting_source, 
                color='red', 
                linestyle='--', 
                linewidth=2,
                label='inserting_source'
            )
        
        axes[i].set_xlabel("Timestamp")
        axes[i].set_ylabel("VOC")
        axes[i].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    print("\n")

IndentationError: expected an indented block after 'if' statement on line 23 (2640873429.py, line 25)

In [ ]:
df_userInputData_smallScale = loadDataFromFile("df_userInputData_smallScale")

In [ ]:
df_userInputData_smallScale

In [ ]:
df_unique_position_count = countPositionOfPollutantSource(df_userInputData_smallScale)
df_unique_position_count

In [ ]:
df_timeSeriesExperiments = loadDataFromFile("df_timeSeriesExperiments")

In [ ]:
df_timeSeriesExperiments

In [ ]:
list_of_userInputData_smallScale_based_on_position = sortUserIputDataToListBasedOfPosition(df_userInputData_smallScale)
list_of_userInputData_smallScale_based_on_position[0]

In [ ]:
for i in range(0,((df_unique_position_count.shape[0]) - 1)):#we will set the starting source as two minutes before the inserting source and 3 minutes source after the inserting source
#check first if all the experiments at start experiment is at least two minutes before inserting source
good_to_merge_start = True
good_to_merge_end = True
for df_exp_based_on_position in list_of_userInputData_smallScale_based_on_position:
    for index, row in df_exp_based_on_position.iterrows(): 
            index_df_small_scale = df_userInputData_smallScale.loc[df_userInputData_smallScale['_id'] == row['_id']].index[0]
            starting_experiment = df_userInputData_smallScale.iloc[index - 1] if index > 0 else None
            removing_source = df_userInputData_smallScale.iloc[index + 1] if index < len(df_userInputData_smallScale) - 1 else None
            if starting_experiment["timestamp_local"] > (df_userInputData_smallScale.iloc[index]["timestamp_local"] - datetime.timedelta(minutes=2)):
                good_to_merge_start = False
            if removing_source["timestamp_local"] > (df_userInputData_smallScale.iloc[index]["timestamp_local"]  + datetime.timedelta(minutes=3)):
                good_to_merge_start = False    
print("Starting experiment is more than two minutes:")
print(good_to_merge_start)
print("Ending experiment is more than two minutes:")
print(good_to_merge_end)
        print_time_series_based_of_position(i,df_userInputData_smallScale,df_timeSeriesExperiments,list_of_userInputData_smallScale_based_on_position)

In [ ]:
list_of_userInputData_smallScale_based_on_position_merged = []

for df_exp_based_on_position in list_of_userInputData_smallScale_based_on_position:
    # Make an explicit copy to avoid SettingWithCopyWarning
    df_exp_based_on_position = df_exp_based_on_position.copy()

    # Create new columns based on 'timestamp_local'
    df_exp_based_on_position["timestamp:Starting Experiment"] = df_exp_based_on_position["timestamp_local"] - datetime.timedelta(minutes=2)
    df_exp_based_on_position["timestamp:Ending Experiment"] = df_exp_based_on_position["timestamp_local"] + datetime.timedelta(minutes=3)

    # Rename original column
    df_exp_based_on_position.rename(columns={'timestamp_local': 'timestamp:Inserting Source'}, inplace=True)

    list_of_userInputData_smallScale_based_on_position_merged.append(df_exp_based_on_position)


In [ ]:
list_of_userInputData_smallScale_based_on_position_merged[0]


In [ ]:
for i in range(0,((df_unique_position_count.shape[0]) - 1)):
        print_time_series_based_of_position(i,df_userInputData_smallScale,df_timeSeriesExperiments,list_of_userInputData_smallScale_based_on_position_merged)